# Interactive PyVista in Google Colab
A minimal, standalone test of browser-side 3D rendering. No tight-focusing repository or GPU required. Run the installation first, then restart the session if prompted and run cells 2–5. Try rotating, zooming and panning the sphere. The HTML export is also useful for a personal website.

In [ ]:
# 1. Install current PyVista + interactive backends
%pip install -q -U pyvista vtk "trame-pyvista[jupyter]"

In [ ]:
# 2. Set up off-screen rendering and report versions
import os
os.environ["PYVISTA_OFF_SCREEN"] = "true"
import pyvista as pv
import vtk
from importlib.metadata import version
from IPython.display import display, HTML
from pathlib import Path

pv.OFF_SCREEN = True
print("PyVista:", pv.__version__)
print("VTK:", vtk.vtkVersion.GetVTKVersion())
print("Trame-PyVista:", version("trame-pyvista"))

def sphere_scene():
    pl = pv.Plotter(notebook=True, off_screen=True, window_size=(800, 550))
    pl.set_background("white")
    pl.add_mesh(pv.Sphere(theta_resolution=64, phi_resolution=64),
                color="royalblue", smooth_shading=True,
                ambient=0.35, specular=0.0)
    pl.add_axes()
    pl.camera_position = "iso"
    return pl

## 3. Interactive inline test
The HTML backend renders the exported scene in the browser without a persistent Trame server. Drag to rotate; scroll to zoom. If the output is blank or static, continue with the standalone export below.

In [ ]:
pl = sphere_scene()
try:
    pl.show(jupyter_backend="html")
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
    print("Try the standalone HTML export in the next section.")

## 4. Standalone HTML fallback
This creates an interactive file suitable for opening in a browser. It also attempts an inline Colab preview. If the preview is blocked by notebook security or browser settings, download the file and open it in a regular browser.

In [ ]:
from html import escape

pl = sphere_scene()
output = Path("pyvista_interactive_sphere.html").resolve()
if hasattr(pl, "trame") and hasattr(pl.trame, "export_html"):
    pl.trame.export_html(str(output))
else:
    pl.export_html(str(output))
pl.close()
print(f"Exported {output.name} ({output.stat().st_size/1024**2:.2f} MB)")
html = output.read_text(encoding="utf-8")
display(HTML(f'<iframe srcdoc="{escape(html, quote=True)}" width="100%" height="550" style="border:0" allowfullscreen></iframe>'))

In [ ]:
# 5. Optional: download the interactive HTML file from Colab
from google.colab import files
files.download(str(output))

### Troubleshooting
If PyVista still reports missing Trame dependencies after installation, restart the session and run cell 2 onward. If VTK reports a missing OpenGL system library, try the optional cell below and restart. An interactive scene in a regular browser is the success criterion even if Colab blocks the embedded viewer.

Documentation: [PyVista Jupyter plotting](https://docs.pyvista.org/user-guide/jupyter/index.html) · [Trame-PyVista](https://github.com/pyvista/trame-pyvista).

In [ ]:
# Optional: run ONLY if VTK reports missing OpenGL/EGL system libraries.
!apt-get update -qq
!apt-get install -y -qq libgl1 libegl1 libgl1-mesa-dri